In [1]:
import os
import weaviate
import weaviate.classes.config as wvc

from dotenv import load_dotenv
from openai import OpenAI

import requests
from bs4 import BeautifulSoup
import tiktoken

In [2]:
load_dotenv(r'.env')

True

### Step 1: get news information from url and chunk the information

In [17]:
def get_content(url):
    response = requests.get(url)

    soup = BeautifulSoup(response.text, 'html.parser')

    for script_or_style in soup(["script", "style", "header", "footer", "nav"]):
        script_or_style.decompose()
        

    text = soup.get_text(separator="\n")

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    clean_text = " ".join(lines)
    return clean_text

In [18]:
def seed_chunks_by_tokens(text, model_name="gpt-4o", chunk_size=500, chunk_overlap=50):

    encoding = tiktoken.encoding_for_model(model_name)
    
    all_tokens = encoding.encode(text)
    
    chunks = []

    for i in range(0, len(all_tokens), chunk_size - chunk_overlap):

        chunk_tokens = all_tokens[i : i + chunk_size]
        chunk_text = encoding.decode(chunk_tokens)
        chunks.append(chunk_text)
        
    return chunks


In [19]:
chunk_size = 100
chunk_overlap = 20
chunks = []
url1 = "https://www.azfamily.com/2024/07/01/evacuation-orders-lifted-boulder-view-fire-near-scottsdale-63-contained/"
url2 = "https://www.kjzz.org/news/2024-06-28/60-forced-to-evacuate-from-boulder-view-fire-in-north-scottsdale"

content1 = get_content(url1)
chunks1 = seed_chunks_by_tokens(content1, chunk_size=chunk_size, chunk_overlap=chunk_overlap)

content2 = get_content(url2)
chunks2 = seed_chunks_by_tokens(content2, chunk_size=chunk_size, chunk_overlap=chunk_overlap)



for c in chunks1:
    chunks.append({'content': c, "source_url": url1})

for c in chunks2:
    chunks.append({'content': c, "source_url": url2})


### Step 2: store chunks into vector database: weaviate

In [7]:
client = weaviate.connect_to_local(
    headers={"X-OpenAI-Api-Key": os.getenv("OPENAI_API_KEY")}
)
client.connect()

In [8]:
collection_name = "NewsChunk"

# client.collections.delete(collection_name)

if not client.collections.exists(collection_name):
    client.collections.create(
        name=collection_name,
        vector_config=[wvc.Configure.Vectors.text2vec_openai(
            name='core', 
            source_properties=['content'],
            model='text-embedding-3-large',
            dimensions=1024
        )],
        properties=[
            wvc.Property(name="content", 
                         data_type=wvc.DataType.TEXT),
            wvc.Property(name="source_url", 
                         data_type=wvc.DataType.TEXT)
        ]
    )
    news_chunks = client.collections.get(collection_name)
    with news_chunks.batch.dynamic() as batch:
        for c in chunks:
            batch.add_object(
                properties={
                    "content": c['content'],
                    "source_url": c['source_url']
                }
            )
    print(f"successfully stored {len(chunks)} data points")
news_chunks = client.collections.use(collection_name)

### Step 3: Define question answering process
1. top k chunks retrieval
2. answer generation

In [9]:
def RAG_Q_A(question, vector_database, LLM_clinet, model_name):
    response = vector_database.query.near_text(
    query=question,
    limit=3)

    retrieved_content = ''
    for obj in response.objects:
        retrieved_content += obj.properties['content'] + '\n'
        # print(retrieved_content)
        
    augmented_question = augmented_question_template.format(question=question, retrieved_content=retrieved_content)

    response = LLM_client.responses.create(
        model = model_name,
        input = augmented_question
        )

    return response.output_text, retrieved_content

In [10]:
augmented_question_template = '''
You will need to answer my question based on the information I give you,

This is the question:
{question}

This is the background information:
{retrieved_content}
'''

LLM_client = OpenAI()
model_name = 'gpt-4.1-mini'

#### Answer question 1

In [11]:
question = "What roads or areas were affected (closed or restricted) during the fire, and which of them were later reopened?"
answer, retrieved_content = RAG_Q_A(question, news_chunks, LLM_client, model_name)

In [12]:
print(answer)

Based on the information provided:

- Roads and areas affected (closed or restricted) during the fire:
  - The north part of the McDowell Sonoran Preserve, including all trails and access points north of Dynamite Boulevard, were closed until further notice.
  - Residents in the evacuation zone from 136th Street to Box Bar Road and Rio Verde Road to Dove Valley were ordered to evacuate.
  - Approximately 60 homes were evacuated.

- Roads and areas later reopened:
  - All trailheads in the McDowell Sonoran Preserve were reopened Monday morning.
  - Bartlett Lake Dam Road and Horseshoe Dam Road were also reopened Monday morning.
  - Evacuation orders for residents from 136th Street to Box Bar Road and Rio Verde Road to Dove Valley were lifted on Saturday afternoon, allowing residents to return.

In summary, the north part of the McDowell Sonoran Preserve (north of Dynamite Boulevard) and trails were closed but later reopened; Bartlett Lake Dam Road and Horseshoe Dam Road were reopened aft

In [13]:
print(retrieved_content)

 out of everyone’s way and help where I can,” said Engelker. The City of Scottsdale notified homeowners that the north part of the McDowell Sonoran Preserve, including all trails and access points north of Dynamite Boulevard, would be closed until further notice. All trailheads were reopened Monday morning. Bartlett Lake Dam Road and Horseshoe Dam Road were reopened Monday morning. Tiffany Davila, spokeswoman with the Department of Forestry and Fire Management, says around 60 homes were evacuated. The fire also
 statuses for those near the fire. This means that residents in the evacuation zone are returning to “READY” status as crews made progress on the fire. On Friday, residents in the area from 136th Street to Box Bar Road and Rio Verde Road to Dove Valley were told to evacuate as the fire grew closer to their neighborhood. Those evacuation orders were lifted on Saturday afternoon. “We were woken up by a police helicopter with spotlights in our area on the radio asking everybody to 

#### Answer question 2

In [14]:
question = "At the beginning of the wildfire (June 28), where did evacuations occur and how many people (or homes) were evacuated?"
answer, retrieved_content = RAG_Q_A(question, news_chunks, LLM_client, model_name)

In [15]:
print(answer)

At the beginning of the wildfire on June 28, evacuations occurred in the Vista Verde community, near the edge of the Tonto National Forest. Approximately 60 people (or homes) were evacuated after Maricopa County emergency personnel ordered evacuations. The evacuated area was east of 136th Street and north of Dove Valley Road, stretching to the boundary of the Tonto National Forest.


In [16]:
print(retrieved_content)

 with the Department of Forestry and Fire Management, says around 60 homes were evacuated. The fire also threatened power lines in the area. “It’s very uneasy knowing that anything could happen at any moment. The wind shifts and fire was moving in one direction last night and then it shifted in the middle of the night, so here we are,” said Engelker. On the night it began, the southeast end of the fire grew significantly, prompting evacuations for part of the Vista Verde community. “
 60 people had left their homes by Friday morning after Maricopa County emergency personnel ordered evacuations for the subdivision on the edge of the Tonto National Forest. The Maricopa County Department of Emergency Management on Friday expanded the list of who needs to get set to evacuate. As of Friday morning, residents between 136th Street to Box Bar Road and Rio Verde to Dove Valley roads should be prepared. Fire officials said they were investigating exactly what sparked the blaze. Nearly 200 firefi